In [ ]:
# In this code, we did the ndQTSA Generation and use deepseek API to filter and format ndQTSA output.
# node7.jsonl is the final subsection file. And gpt_output/QTSA_new1.jsonl is the final ndQTSA file.

In [ ]:
# ndQTSA Generation

import requests
import json, os
from pydantic import BaseModel
from typing import Optional, List
import time
from pydantic import BaseModel
from typing import Optional
from openai import OpenAI

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

OUTPUT_DIR = "gpt_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def call_llm(system_prompt: str,
             user_prompt: str,
             max_new_tokens: int = 1000,
             temperature: float = 0.1,
             model: str = "gpt-4.1-mini") -> str:
    try:
        #print(f"🌐 activate {model} by API (temperature={temperature})...")

        kwargs = dict(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            stream=False
        )

        if "gpt-5" in model or model.startswith("o1"):   
            kwargs["max_completion_tokens"] = max_new_tokens
        else:  
            kwargs["max_tokens"] = max_new_tokens

        response = client.chat.completions.create(**kwargs)

        #print(f"✅ finish activating model with API, token: {response.usage.total_tokens}")

        return response.choices[0].message.content

    except Exception as e:
        print(f"❌ Failed to use API: {e}")
        return None

# Set up prompt for multi-agents
# 1. Question Generator Agent (Ψ_Q)
system_prompt_agent_q = """You are an experienced professor of radio frequency integrated circuit design. Your task is to generate a specific, in-depth, and pedagogically valuable question based on the provided textbook content.

CRITICAL REQUIREMENTS:
- The question MUST be strictly based on and answerable from the provided text content ONLY.
- The question should test understanding, not just recall. It can be about conceptual understanding, formula derivation, circuit analysis, or design methodology.
- The question should be clear, unambiguous, and self-contained (does not require external knowledge).
- Output ONLY the question itself, without any explanations, answers, or additional text.
- Avoid the words specialized in pointing figures, such as "as shown in Figure X.Y"
- Do not include the answer of this question.
"""

# 2. Answer Generator Agent (Ψ_A)
system_prompt_agent_a = """You are a meticulous and patient radio frequency integrated circuit design tutor. You MUST follow these steps STRICTLY:

1.  **THINKING PROCESS** (<think> tags): 
    - First, conduct a thorough, step-by-step reasoning process inside <think> and </think> tags.
    - This is your internal monologue. Analyze the problem in detail: break down the question, identify key concepts, recall and summarize relevant principles, formulas, or methodologies from the provided reference text.
    - Plan your solution approach explicitly. Explain *why* you are taking each step, don't just list steps. This section should be comprehensive and exploratory.
2.  **STRUCTURED SOLUTION**: 
    - After your internal thinking, provide a formal, well-organized, and educational solution outside the think tags.
    - This section should translate your reasoning from the <think> section into a clear, logical narrative. Show all necessary calculations and derivations step by step, explaining the rationale behind each step based on the reference text.
    - Use headings, bullet points, or numbered steps for clarity. Ensure this part is detailed enough for a learner to follow the entire logic.
3.  **FINAL ANSWER** (<answer> tags): 
    - Conclude with a concise and precise final answer, wrapped in <answer> and </answer> tags.
    - This section should be brief, containing only the final numerical result, key conclusion, or summarized finding. It should not contain explanations or reasoning, which belong in the previous sections.

RULES:
- Your response MUST be based EXCLUSIVELY on the provided reference text. Do not use external knowledge.
- When applying concepts, you may reference equations from the text by describing their content (e.g., "using the equation for power gain that relates transconductance and load resistance"), but do not use equation numbers or figure references.
- The <think> and STRUCTURED SOLUTION sections must be substantially detailed and form the bulk of your response.
- The <answer> section must be succinct, containing only the final answer.
- Write in clear, academic English.
"""

# 3. Post-processor Agent (Ψ_P)
system_prompt_agent_p = """You are a quality assurance specialist for AI training data. 
Your primary responsibility is to minimally edit the raw QTSA outputs from Agents Q and A, 
ensuring valid formatting and JSON compliance, while preserving their content as faithfully as possible.

CRITICAL REQUIREMENTS:
1. **Minimal Editing**: 
   - Do NOT rewrite or paraphrase unless absolutely necessary for clarity or format.
   - Retain the original wording, technical details, equations, and reasoning from Agents Q and A.
   - If the raw outputs contain errors, only fix them in a minimal way (e.g., unmatched tags, broken equations).

2. **Output Format**: 
   - Output ONLY a valid JSON object. 
   - Do NOT wrap in code blocks (no ```json), no extra text, no explanations.

3. **Format Validation**: 
   - Ensure the JSON object contains all four QTSA components:
     - "question": the refined question
     - "thinking": the thinking process with <think>...</think>
     - "solution": the structured solution
     - "answer": the final answer with <answer>...</answer>
   - Check XML tags (<think>, <answer>) are properly opened and closed.

4. **Decontextualization**: 
   - Remove direct references to the original textbook source.
   - Example: Replace "according to Equation (Z)" with "according to the following equation".
   - Example: Replace "see Table A.B" with "see the following table".

5. **Refinement**: 
   - Only refine for grammar, clarity, and structure if absolutely necessary.
   - Do not shorten, simplify, or change the technical meaning of the content.

OUTPUT FORMAT (pure JSON only):
{
  "question": "...",
  "thinking": "<think> ... </think>",
  "solution": "...",
  "answer": "<answer> ... </answer>"
}

IMPORTANT: 
- Your output must be directly parseable by json.loads().
- Do NOT include markdown code blocks or any text outside the JSON.
"""


class Node(BaseModel):
    id: str  
    content: str  

class QTSAPair(BaseModel):
    question: str
    thinking: str
    solution: str
    answer: str
    node_id: str  
    sample_id: int
    raw_question: Optional[str] = None
    raw_answer: Optional[str] = None
    processed_output: Optional[str] = None

class AgentOutput(BaseModel):
    agent_q_output: str
    agent_a_output: str
    agent_p_output: str
    
# 3 agents

def generate_question(node: Node, sample_id: int = 0) -> str:
    """Agent Ψ_Q: Generate question from node content"""
    
    question_strategies = [
        "Ask a question that explores the fundamental concepts:",
        "Formulate a question requiring mathematical analysis or derivation:",
        "Pose a design-oriented question about implementation:",
        "Create an analytical question about performance aspects:",
        "Develop a question about practical applications or limitations:"
    ]

    strategy_index = (sample_id - 1) % len(question_strategies)
    strategy = question_strategies[strategy_index]
    
    user_prompt = f"""TEXTBOOK CONTENT:
{node.content}

{strategy}
If the content doesn't support this approach, adapt to a suitable perspective while maintaining technical depth."""
    
    question = call_llm(system_prompt_agent_q, user_prompt, temperature=1.0, max_new_tokens=500, model="gpt-4.1-mini")
    return question

def generate_answer(node: Node, question: str) -> str:
    """Agent Ψ_A: Generate answer with thinking process"""
    user_prompt = f"""REFERENCE TEXT:
{node.content}

QUESTION TO ANSWER:
{question}

Please provide a complete response following the required format."""
    
    answer = call_llm(system_prompt_agent_a, user_prompt, temperature=1, max_new_tokens=10000, model="gpt-5-mini")
    #if answer:
        #print(f"✅ Agent A 's answer: {answer[:100]}...")
    return answer

def post_process_data(raw_question: str, raw_answer: str) -> str:
    """Agent Ψ_P: Clean and format the QTSA data"""
    user_prompt = f"""RAW GENERATED DATA:
Question: {raw_question}
Answer: {raw_answer}

Please process this data according to the instructions."""
    
    processed_output = call_llm(system_prompt_agent_p, user_prompt, temperature=0.0, max_new_tokens=11000, model="gpt-4.1-mini")
    #if processed_output:
        #print(f"✅ Agent P 's answer: {processed_output[:100]}...")
    return processed_output


def process_single_node(node: Node, sample_id: int = 1) -> Optional[QTSAPair]:
    """process a single node, output entire QTSA and original output"""
    #print(f"Processing node {node.id}, sample #{sample_id + 1}")
    #print("=" * 60)

    # save the original output of each agents
    agent_outputs = AgentOutput(
        agent_q_output="",
        agent_a_output="", 
        agent_p_output=""
    )
    
    # Step 1: Generate question
    raw_question = generate_question(node, sample_id)
    if not raw_question:
        print(f"Failed to generate question for node {node.id}")
        return None
    agent_outputs.agent_q_output = raw_question
    
    # Step 2: generate answer
    raw_answer = generate_answer(node, raw_question)
    if not raw_answer:
        print(f"Failed to generate answer for node {node.id}")
        return None
    agent_outputs.agent_a_output = raw_answer
    
    # step 3: Post-process
    processed_output = post_process_data(raw_question, raw_answer)
    agent_outputs.agent_p_output = processed_output or ""
    
    try:
        if processed_output and processed_output.strip().startswith('{'):
            data_dict = json.loads(processed_output)
            qtsa_pair = QTSAPair(
                question=data_dict.get("question", raw_question),
                thinking=data_dict.get("thinking", ""),
                solution=data_dict.get("solution", ""),
                answer=data_dict.get("answer", ""),
                node_id=node.id,
                sample_id=sample_id,
                raw_question=raw_question,
                raw_answer=raw_answer,
                processed_output=processed_output
            )
        else:
           
            qtsa_pair = QTSAPair(
                question=raw_question,
                thinking=raw_answer,
                solution=raw_answer,
                answer=raw_answer,
                node_id=node.id,
                sample_id=sample_id,
                raw_question=raw_question,
                raw_answer=raw_answer,
                processed_output=processed_output or "No JSON output"
            )
        
        #print(f"✅ paragraph {node.id} is finished ")

        save_agent_outputs(agent_outputs, node.id, sample_id)
        
        return qtsa_pair
        
    except json.JSONDecodeError as e:
        print(f"❌ Fail to parse the file: {e}")
        return QTSAPair(
            question=raw_question,
            thinking=raw_answer,
            solution=raw_answer,
            answer=raw_answer,
            node_id=node.id,
            sample_id=sample_id,
            raw_question=raw_question,
            raw_answer=raw_answer,
            processed_output=f"JSON Parse Error: {e}"
        )

def save_agent_outputs(agent_outputs: AgentOutput, node_id: str, sample_id: int):
    agent_filename = f"{OUTPUT_DIR}/QTSA_agent.jsonl"                # output file
    
    output_data = {
        "node_id": node_id,
        "sample_id": sample_id,
        "agent_q_output": agent_outputs.agent_q_output,
        "agent_a_output": agent_outputs.agent_a_output,
        "agent_p_output": agent_outputs.agent_p_output
    }
    
    with open(agent_filename, 'a', encoding='utf-8') as f:
        f.write(json.dumps(output_data, ensure_ascii=False) + '\n')
    
    #print(f"💾 Original outputs are saved as: {agent_filename}")

def process_multiple_samples(node: Node, num_samples: int = 3) -> List[QTSAPair]:
    """Process multiple samples for a single node"""
    results = []
    
    print(f"Processing node {node.id} with {num_samples} samples...")
    
    for sample_id in range(1, num_samples + 1): 
        result = process_single_node(node, sample_id)
        if result:
            results.append(result)
            print(f"  Sample #{sample_id} completed")
        else:
            print(f"  Sample #{sample_id} failed")
        
        # Add delay between samples
        if sample_id < num_samples - 1:
            time.sleep(2)
    
    #print(f"Node {node.id} completed: {len(results)}/{num_samples} samples successful")
    return results

def save_results(results: List[QTSAPair], filename: str = None):
    """Save results to JSONL file"""
    if not results:
        print("No results to save")
        return
    
    with open(filename, 'a', encoding='utf-8') as f:
        for result in results:
            clean_data = {
                "node_id": result.node_id,
                "sample_id": result.sample_id,
                "question": result.question,
                "thinking": result.thinking,
                "solution": result.solution,
                "answer": result.answer
            }
            f.write(json.dumps(clean_data, ensure_ascii=False) + '\n')
    
    #print(f"Results saved to {filename}")

# Test multiple samples for one node
if __name__ == "__main__":
    nodes = []
    try:
        with open('node7.jsonl', 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line.strip())
                nodes.append(Node(id=data["id"], content=data["text"]))
        
        print(f"Loaded {len(nodes)} nodes")

        num_samples = 5   

        for idx, selected_node in enumerate(nodes):
            #print(f"\n=== Processing node #{idx + 1}: {selected_node.id} ===")

            results = process_multiple_samples(selected_node, num_samples)

            if results:
                filename = os.path.join(
                    OUTPUT_DIR, "QTSA.jsonl"          # output
                )
                save_results(results, filename)

                print(f"Generated {len(results)} QTSA pairs -> saved to {filename}")
            else:
                print("No results were generated for this node")

        print(f"All processing completed. Files saved to {OUTPUT_DIR}")

    except FileNotFoundError:
        print("Error: node7.jsonl file not found")
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

In [ ]:
import json

def check_missing_combinations_simple_corrected(file_path):
    # Collect all existing combinations
    existing_combinations = set()
    
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            data = json.loads(line.strip())
            node_id = data.get('node_id')
            sample_id = data.get('sample_id')
            
            if node_id is not None and sample_id is not None:
                # Key fix: convert string node_id to integer
                existing_combinations.add((int(node_id), sample_id))
    
    # Generate all expected combinations
    expected_combinations = {(node_id, sample_id) 
                           for node_id in range(1, 1109) 
                           for sample_id in range(1, 6)}
    
    # Find missing combinations
    missing_combinations = expected_combinations - existing_combinations
    
    print(f"Total data rows: {len(existing_combinations)}")
    print(f"Expected combinations: {len(expected_combinations)}")
    print(f"Missing combinations: {len(missing_combinations)}")
    
    if missing_combinations:
        print("\nMissing node_id and sample_id combinations:")
        
        # Display grouped by node_id
        current_node = None
        missing_samples = []
        
        for node_id, sample_id in sorted(missing_combinations):
            if node_id != current_node:
                if current_node is not None:
                    print(f"node_id {current_node}: missing sample_id {missing_samples}")
                current_node = node_id
                missing_samples = [sample_id]
            else:
                missing_samples.append(sample_id)
        
        # Print the last group
        if current_node is not None:
            print(f"node_id {current_node}: 缺失 sample_id {missing_samples}")
    else:
        print("\nNo missing combinations found!")

# 使用示例
file_path = "gpt_output/QTSA.jsonl"
check_missing_combinations_simple_corrected(file_path)

In [ ]:
import json

def check_missing_combinations_simple_corrected(file_path):
    # Collect all existing combinations
    existing_combinations = set()
    
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            data = json.loads(line.strip())
            node_id = data.get('node_id')
            sample_id = data.get('sample_id')
            
            if node_id is not None and sample_id is not None:
                # Key fix: convert string node_id to integer
                existing_combinations.add((int(node_id), sample_id))
    
    # Generate all expected combinations
    expected_combinations = {
        (node_id, sample_id) 
        for node_id in range(1, 1109) 
        for sample_id in range(1, 6)
    }
    
    # Find missing combinations
    missing_combinations = expected_combinations - existing_combinations
    
    print(f"Total data rows: {len(existing_combinations)}")
    print(f"Expected combinations: {len(expected_combinations)}")
    print(f"Missing combinations: {len(missing_combinations)}")
    
    if missing_combinations:
        print("\nMissing node_id and sample_id combinations:")
        
        # Display grouped by node_id
        current_node = None
        missing_samples = []
        
        for node_id, sample_id in sorted(missing_combinations):
            if node_id != current_node:
                if current_node is not None:
                    print(f"node_id {current_node}: missing sample_id {missing_samples}")
                current_node = node_id
                missing_samples = [sample_id]
            else:
                missing_samples.append(sample_id)
        
        # Print the last group
        if current_node is not None:
            print(f"node_id {current_node}: missing sample_id {missing_samples}")
    else:
        print("\nNo missing combinations found!")

# Example usage
file_path = "gpt_output/QTSA_agent.jsonl"
check_missing_combinations_simple_corrected(file_path)

In [ ]:
import json
import re

qtsa_file = "gpt_output/QTSA.jsonl"
agent_file = "gpt_output/QTSA_agent.jsonl"
output_file = "gpt_output/QTSA_new.jsonl"

agent_pairs = set()
with open(agent_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        agent_pairs.add((item["node_id"], item["sample_id"]))

new_data = []
problem_entries = []

think_pattern = re.compile(r"<think>.*?</think>", re.DOTALL)
answer_pattern = re.compile(r"<answer>.*?</answer>", re.DOTALL)

with open(qtsa_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        key = (item["node_id"], item["sample_id"])

        if key not in agent_pairs: 
            full_text = json.dumps(item, ensure_ascii=False)

            think_match = think_pattern.search(full_text)
            answer_match = answer_pattern.search(full_text)

            if think_match and answer_match:
                think_text = think_match.group(0)
                answer_text = answer_match.group(0)

                solution_text = full_text[
                    full_text.find(think_text) + len(think_text) : full_text.find(answer_text)
                ]

                solution_text = solution_text.replace("STRUCTURED SOLUTION", "").strip()

                item["thinking"] = think_text
                item["solution"] = solution_text
                item["answer"] = answer_text

                problem_entries.append(key)

        new_data.append(item)

print("fixed sample (node_id, sample_id):")
for key in problem_entries:
    print(key)

# save new file
with open(output_file, "w", encoding="utf-8") as f:
    for item in new_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\nFinsih fixing and results are saved as {output_file}")


In [ ]:
import json

input_file = "gpt_output/QTSA_new.jsonl"
output_file = "gpt_output/QTSA_new1.jsonl"

with open(input_file, 'r', encoding='utf-8') as f_in, \
     open(output_file, 'w', encoding='utf-8') as f_out:
    
    for i, line in enumerate(f_in, 1):
        data = json.loads(line)
        data['id'] = i
        f_out.write(json.dumps(data, ensure_ascii=False) + '\n')

print("Finished！")